# Climate Energy Consumption Prediction
A feedforward neural network (Keras/TensorFlow) that predicts energy consumption from climate and time features.

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

## Step 2: Load the Dataset

In [3]:
df = pd.read_csv("climate_energy.csv")
print(df.head())

             timestamp    location  temperature  energy_consumption
0  2024-01-01 00:00:00     Chennai         21.1               42.27
1  2024-01-01 01:00:00       Salem         20.8               41.64
2  2024-01-01 02:00:00     Madurai         21.7               41.88
3  2024-01-01 03:00:00  Coimbatore         19.6               43.51
4  2024-01-01 04:00:00  Coimbatore         20.9               42.29


## Step 3: Dataset Information

In [5]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   timestamp           2000 non-null   object 
 1   location            2000 non-null   object 
 2   temperature         2000 non-null   float64
 3   energy_consumption  2000 non-null   float64
dtypes: float64(2), object(2)
memory usage: 62.6+ KB
None
       temperature  energy_consumption
count  2000.000000         2000.000000
mean     26.476500           66.403965
std       4.153513           21.389659
min      16.700000           20.560000
25%      23.200000           47.997500
50%      26.500000           65.225000
75%      29.700000           82.810000
max      36.600000          124.310000


## Step 4: Feature Engineering
Convert timestamp into numerical features.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month

df.drop("timestamp", axis=1, inplace=True)

print(df.head())

## Step 5: Encode Location

Example mapping:

- Chennai -> 0
- Coimbatore -> 1
- Madurai -> 2
- Salem -> 3

In [7]:
encoder = LabelEncoder()

df["location"] = encoder.fit_transform(df["location"])

print(df.head())

             timestamp  location  temperature  energy_consumption
0  2024-01-01 00:00:00         0         21.1               42.27
1  2024-01-01 01:00:00         3         20.8               41.64
2  2024-01-01 02:00:00         2         21.7               41.88
3  2024-01-01 03:00:00         1         19.6               43.51
4  2024-01-01 04:00:00         1         20.9               42.29


## Step 6: Split Features and Target

In [11]:
X = df.drop("energy_consumption", axis=1)
y = df["energy_consumption"]

## Step 7: Normalize Features

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ValueError: could not convert string to float: '2024-01-01 00:00:00'

## Step 8: Train-Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

NameError: name 'X_scaled' is not defined

## Step 9: Build Feedforward Neural Network

In [15]:
model = Sequential()

model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],))) #input layer

model.add(Dense(8, activation='relu')) # hidden layer

model.add(Dense(1)) #output layer

NameError: name 'X_train' is not defined

## Step 10: Compile Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

## Step 11: View Model Summary

In [ ]:
model.summary()

## Step 12: Train the Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

## Step 13: Evaluate Model

In [ ]:
loss, mae = model.evaluate(X_test, y_test)

print("Test Loss :", loss)
print("Test MAE  :", mae)

## Step 14: Make Predictions

In [ ]:
predictions = model.predict(X_test)

print(predictions[:5])

## Step 15: Calculate Performance Metrics

In [ ]:
mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

mae = mean_absolute_error(y_test, predictions)

r2 = r2_score(y_test, predictions)

print("Mean Squared Error :", mse)
print("Root Mean Squared Error :", rmse)
print("Mean Absolute Error :", mae)
print("R2 Score :", r2)

## Step 16: Predict New Climate Data

Suppose we want to predict energy consumption for:

- Location = Chennai
- Temperature = 35°C
- Hour = 14
- Day = 5
- Month = 1

In [ ]:
new_data = pd.DataFrame({
    "location":[encoder.transform(["Chennai"])[0]],
    "temperature":[35],
    "hour":[14],
    "day":[5],
    "month":[1]
})

new_scaled = scaler.transform(new_data)

prediction = model.predict(new_scaled)

print("Predicted Energy Consumption:", prediction[0][0])

## Step 17: Plot Training Loss

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history["loss"], label="Training Loss")

plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.show()

## Step 18: Plot Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(y_test, predictions)

plt.xlabel("Actual Energy Consumption")

plt.ylabel("Predicted Energy Consumption")

plt.title("Actual vs Predicted")

plt.show()

## Step 19: Prediction Error Distribution

In [ ]:
errors = y_test.values - predictions.flatten()

plt.figure(figsize=(8,5))

plt.hist(errors, bins=20)

plt.xlabel("Prediction Error")

plt.ylabel("Frequency")

plt.title("Prediction Error Distribution")

plt.show()